# Semana 12: Redes neuronales: la red 2-2-1 de la clase

**Introducción a la Ciencia de Datos** · Dudbil Olvasada Pabón Riaño · UNAB

Cuaderno guiado. Cada celda corresponde a un ejercicio de la guía de laboratorio; los resultados deben coincidir con sus puntos de control. Los datos se leen desde el repositorio del curso.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
BASE = "https://raw.githubusercontent.com/dudbilopr/Introduccion-a-la-ciencia-de-datos/main/Semanas/Datos/"

La red de la clase en Excel, ahora en Python. AND es un ejemplo para calcular a mano; al final se aplica la misma idea a datos reales de churn.

In [ ]:
sig = lambda z: 1 / (1 + np.exp(-z))
P0 = dict(w11=.2, w12=-.3, b1=-.4, w21=.3, w22=.2, b2=-.1, w31=.4, w32=.5, bs=-.3)
AND = [(0, 0, 0), (0, 1, 0), (1, 0, 0), (1, 1, 1)]
eta = 0.5

def paso(P, x1, x2, y):
    z1 = P["w11"] * x1 + P["w12"] * x2 + P["b1"]; a1 = sig(z1)
    z2 = P["w21"] * x1 + P["w22"] * x2 + P["b2"]; a2 = sig(z2)
    zs = P["w31"] * a1 + P["w32"] * a2 + P["bs"]; yh = sig(zs)
    L = -(y * np.log(yh) + (1 - y) * np.log(1 - yh))
    ds = yh - y                       # BCE + sigmoide: el factor yh(1 - yh) se cancela
    d1 = ds * P["w31"] * a1 * (1 - a1)
    d2 = ds * P["w32"] * a2 * (1 - a2)
    g = dict(w31=ds * a1, w32=ds * a2, bs=ds, w11=d1 * x1, w12=d1 * x2, b1=d1, w21=d2 * x1, w22=d2 * x2, b2=d2)
    return dict(zs=zs, yh=yh, L=L, ds=ds, d1=d1, d2=d2), g

## 1. Época 1 con SGD en línea
Debe coincidir con la tabla de la clase.

In [ ]:
P = dict(P0)
for k, (x1, x2, y) in enumerate(AND, 1):
    v, g = paso(P, x1, x2, y)
    P = {n: P[n] - eta * g[n] for n in P}
    print(f"M{k}: zs={v['zs']:.4f}  ŷ={v['yh']:.4f}  L={v['L']:.4f}  δs={v['ds']:.4f}  δ1={v['d1']:.4f}  δ2={v['d2']:.4f}")
print("P4:", {n: round(x, 4) for n, x in P.items()})

## 2. En línea contra por lotes

In [ ]:
def entrenar(modo, epocas=2000):
    P, curva, primera = dict(P0), [], None
    for e in range(1, epocas + 1):
        if modo == "linea":
            for m in AND:
                P = {n: P[n] - eta * g for n, g in paso(P, *m)[1].items()}
        else:
            gs = [paso(P, *m)[1] for m in AND]
            P = {n: P[n] - eta * np.mean([g[n] for g in gs]) for n in P}
        sal = [paso(P, *m)[0] for m in AND]
        curva.append(np.mean([s["L"] for s in sal]))
        if primera is None and all((s["yh"] >= .5) == bool(m[2]) for s, m in zip(sal, AND)):
            primera = e
    return curva, primera
cl, el = entrenar("linea"); cb, eb = entrenar("lote")
print("Clasifica bien AND desde la época: en línea", el, " por lotes", eb)
print("Pérdida tras 2.000 épocas: en línea", round(cl[-1], 4), " por lotes", round(cb[-1], 4))
plt.plot(cl[:400], label="En línea"); plt.plot(cb[:400], label="Por lotes"); plt.xlabel("Época"); plt.ylabel("Pérdida media"); plt.legend(); plt.show()

## 3. Una sola neurona frente a XOR

In [ ]:
from sklearn.linear_model import LogisticRegression
Xx = np.array([[0, 0], [0, 1], [1, 0], [1, 1]]); yx = np.array([0, 1, 1, 0])
lx = LogisticRegression(C=1e6, max_iter=5000).fit(Xx, yx)
print("Exactitud:", lx.score(Xx, yx), " probabilidades:", lx.predict_proba(Xx)[:, 1].round(3))

## 4. Con datos reales: churn sin duplicados

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import recall_score
df = pd.read_csv(BASE + "churn_telecom.csv"); df.columns = [" ".join(c.split()) for c in df.columns]
u = df.drop_duplicates(); y, X = u["Churn"], u.drop(columns="Churn")
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(Xtr, ytr)
red = make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(8,), activation="logistic", max_iter=3000, random_state=42)).fit(Xtr, ytr)
for nombre, m in [("Logística", lr), ("Red, 8 ocultas", red)]:
    print(nombre, " exactitud", round(m.score(Xte, yte), 3), " sensibilidad", round(recall_score(yte, m.predict(Xte)), 3))

## Declaración de uso de IA

Si usó una herramienta de IA, escriba aquí: cuál, para qué, el prompt, la versión del modelo y qué verificó usted.